<a href="https://colab.research.google.com/github/SRIKRISHNA1407627/BIS_LAB_1BM24CS151/blob/main/WEEK_05_GWO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

class EVGreyWolfOptimizer:
    def __init__(self, num_candidates, demand, distance_matrix, fixed_costs,
                 num_wolves=30, max_iter=200):
        """
        Parameters:
        - num_candidates: Total potential EV station locations (m)
        - demand: Array of charging demand at each user node (n)
        - distance_matrix: Shape (m, n) matrix of distances from candidate i to user j
        - fixed_costs: Array of setup costs for each candidate station (m)
        - num_wolves: Pack size (population)
        - max_iter: Maximum optimization iterations
        """
        self.m = num_candidates
        self.n = len(demand)
        self.demand = demand
        self.dist_mat = distance_matrix
        self.fixed_costs = fixed_costs

        self.num_wolves = num_wolves
        self.max_iter = max_iter

    def fitness(self, wolf_position):
        """Calculates total costs. Lower cost = superior fitness."""
        # Convert continuous position values into a discrete binary placement vector
        placement = (wolf_position > 0.5).astype(int)

        # Penalty rule: Must select at least one site
        if np.sum(placement) == 0:
            return float('inf')

        # 1. Fixed Capital Cost Evaluation
        setup_cost = np.sum(placement * self.fixed_costs)

        # 2. Total User Travel Cost Allocation
        active_indices = np.where(placement == 1)[0]
        active_distances = self.dist_mat[active_indices, :]
        min_distances = np.min(active_distances, axis=0)
        travel_cost = np.sum(min_distances * self.demand)

        return setup_cost + travel_cost

    def optimize(self):
        # Initialize the positions of the wolf pack uniformly between [0, 1]
        positions = np.random.uniform(0, 1, (self.num_wolves, self.m))

        # Initialize alpha, beta, and delta positions and their fitness scores
        alpha_pos = np.zeros(self.m)
        alpha_score = float("inf")

        beta_pos = np.zeros(self.m)
        beta_score = float("inf")

        delta_pos = np.zeros(self.m)
        delta_score = float("inf")

        # Main Optimization Loop
        for t in range(self.max_iter):

            # Step 1: Update Alpha, Beta, and Delta based on hierarchy fitness
            for i in range(self.num_wolves):
                # Bound check to keep pack positions within hypercube limits
                positions[i] = np.clip(positions[i], 0, 1)

                fit = self.fitness(positions[i])

                # Update leadership hierarchy
                if fit < alpha_score:
                    delta_score = beta_score
                    delta_pos = beta_pos.copy()
                    beta_score = alpha_score
                    beta_pos = alpha_pos.copy()
                    alpha_score = fit
                    alpha_pos = positions[i].copy()

                elif fit < beta_score:
                    delta_score = beta_score
                    delta_pos = beta_pos.copy()
                    beta_score = fit
                    beta_pos = positions[i].copy()

                elif fit < delta_score:
                    delta_score = fit
                    delta_pos = positions[i].copy()

            # Linearly decrease the exploration parameter 'a' from 2 down to 0
            a = 2 - t * (2 / self.max_iter)

            # Step 2: Encircling & Hunting - Update positions of all omega wolves
            for i in range(self.num_wolves):
                for j in range(self.m):

                    # Update equation guided by Alpha
                    r1, r2 = np.random.rand(), np.random.rand()
                    A1 = 2 * a * r1 - a
                    C1 = 2 * r2
                    D_alpha = abs(C1 * alpha_pos[j] - positions[i, j])
                    X1 = alpha_pos[j] - A1 * D_alpha

                    # Update equation guided by Beta
                    r1, r2 = np.random.rand(), np.random.rand()
                    A2 = 2 * a * r1 - a
                    C2 = 2 * r2
                    D_beta = abs(C2 * beta_pos[j] - positions[i, j])
                    X2 = beta_pos[j] - A2 * D_beta

                    # Update equation guided by Delta
                    r1, r2 = np.random.rand(), np.random.rand()
                    A3 = 2 * a * r1 - a
                    C3 = 2 * r2
                    D_delta = abs(C3 * delta_pos[j] - positions[i, j])
                    X3 = delta_pos[j] - A3 * D_delta

                    # Compute new average coordinate point for the current dimension
                    positions[i, j] = (X1 + X2 + X3) / 3

        # Return final optimal discrete configuration vector and its matching operational cost
        optimal_binary_layout = (alpha_pos > 0.5).astype(int)
        return optimal_binary_layout, alpha_score

# --- Execution Testbed ---
if __name__ == "__main__":
    np.random.seed(42)

    # Grid Parameters
    CANDIDATES = 10
    DEMAND_POINTS = 30

    # Synthesize Dataset
    mock_demand = np.random.randint(10, 100, size=DEMAND_POINTS)
    mock_distances = np.random.uniform(0.5, 15.0, size=(CANDIDATES, DEMAND_POINTS))
    mock_fixed_costs = np.random.uniform(50000, 150000, size=CANDIDATES)

    # Execute Optimizer
    gwo_solver = EVGreyWolfOptimizer(
        num_candidates=CANDIDATES,
        demand=mock_demand,
        distance_matrix=mock_distances,
        fixed_costs=mock_fixed_costs,
        num_wolves=30,
        max_iter=150
    )

    optimal_layout, minimal_cost = gwo_solver.optimize()

    print("=== EV Station GWO Optimizer Output ===")
    print(f"Optimal Station Activation Strategy: {optimal_layout}")
    print(f"Active Station Indices: {np.where(optimal_layout == 1)[0]}")
    print(f"Minimized Combined Cost Score: {minimal_cost:,.2f}")


=== EV Station GWO Optimizer Output ===
Optimal Station Activation Strategy: [0 0 0 0 0 0 0 0 0 1]
Active Station Indices: [9]
Minimized Combined Cost Score: 78,149.08
